In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
import asyncio
import sys

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server from ./resources/2.1_mcp_server.py

In [6]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
    )

In [7]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=str(prompt)
    )

In [9]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config # type: ignore
    ) 

In [10]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='ced5b134-1296-4e37-8585-a88a7dc78a40'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 273, 'total_tokens': 429, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EABUL0jVmQE8AbynFJWcMJl9BqbRU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fdba0-5c64-76a0-a8d9-a94c140dd69a-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters'}, 'id': 'call_rg0k3H9SEF51pCnbKiWEg4pj', 'type': 'tool_call'}], invalid_tool_calls=[], us

In [17]:
import textwrap

text = response["messages"][-1].content
wrapped = "\n\n".join(textwrap.fill(p, width=110) for p in text.split("\n\n"))
print(wrapped)

Here’s what the LangChain MCP Adapters library is and how it’s typically used.

What it is - A bridge that makes Anthropic Model Context Protocol (MCP) tools compatible with LangChain and
LangGraph. - It acts as a translation layer so you can treat MCP tool servers as first-class LangChain tools
and orchestrate them in LangChain agents and LangGraph workflows. - There is a Python package (langchain-mcp-
adapters) and a JavaScript/TypeScript version for LangChain.js (@langchain/mcp-adapters).

Key features - Convert MCP tools into LangChain/LangGraph-ready tools. - A client that can connect to multiple
MCP servers and load tools from them. - Enables chaining or composing tools from different MCP servers in a
single LangChain/LangGraph agent.

Why you’d use it - If you already have existing MCP-enabled tool servers (e.g., Box, Math, CRM tools) and want
to use them inside LangChain workflows without writing custom glue code for each server. - To access hundreds
of MCP tools across multipl

## Online MCP

In [ ]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "--with", "mcp<2",  # pin uvx to install with an older, compatible mcp alongside it
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [20]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [21]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='d5f030df-d4c1-4afb-91bd-d2f4f8e10c16'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 219, 'prompt_tokens': 295, 'total_tokens': 514, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EABg4n0otU0oS8tKW2eAxXMMM7Pqn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fdbab-75bd-7963-89cd-52407e33fe5c-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'call_L0IrB6F8FFgwRIqlmlldk3I8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens':

In [22]:
pprint(response["messages"][-1].content)

("It's Friday, August 7, 2026, 06:01:19 in New York (Eastern Daylight Time, "
 'UTC-4). Want me to convert this to another time zone?')
